In [1]:
import pandas as pd
import numpy as np

DATA_PATH = r"D:\PS26-PAPER1\data\processed\cicids2017_final.csv"

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("FEATURE ANALYSIS")
print("=" * 70)

print("Dataset shape:", df.shape)

feature_cols = [
    col for col in df.columns
    if col != "Label"
]

X = df[feature_cols]

print("Number of features:", len(feature_cols))

FEATURE ANALYSIS
Dataset shape: (2805750, 79)
Number of features: 78


In [2]:
constant_features = [
    col
    for col in feature_cols
    if X[col].nunique() <= 1
]

print("Constant features:")
print(constant_features)

print(
    "\nNumber of constant features:",
    len(constant_features)
)

Constant features:
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Number of constant features: 8


In [3]:
variance = X.var()

near_zero_variance = (
    variance[
        variance < 1e-10
    ]
    .sort_values()
)

print("Near-zero variance features:")
print(near_zero_variance)

print(
    "\nNumber of near-zero variance features:",
    len(near_zero_variance)
)

Near-zero variance features:
Bwd PSH Flags           0.0
Bwd URG Flags           0.0
Fwd Avg Bytes/Bulk      0.0
Fwd Avg Packets/Bulk    0.0
Fwd Avg Bulk Rate       0.0
Bwd Avg Bytes/Bulk      0.0
Bwd Avg Packets/Bulk    0.0
Bwd Avg Bulk Rate       0.0
dtype: float64

Number of near-zero variance features: 8


In [4]:
feature_stats = pd.DataFrame({
    "dtype": X.dtypes,
    "unique_values": X.nunique(),
    "mean": X.mean(),
    "std": X.std(),
    "min": X.min(),
    "max": X.max()
})

display(
    feature_stats.sort_values(
        "unique_values"
    ).head(30)
)

,dtype,unique_values,mean,std,min,max
Bwd PSH Flags,int64,1,0.000000,0.000000e+00,0.000000e+00,0.0
Fwd Avg Bulk Rate,int64,1,0.000000,0.000000e+00,0.000000e+00,0.0
Bwd Avg Bytes/Bulk,int64,1,0.000000,0.000000e+00,0.000000e+00,0.0
Bwd Avg Packets/Bulk,int64,1,0.000000,0.000000e+00,0.000000e+00,0.0
Fwd Avg Packets/Bulk,int64,1,0.000000,0.000000e+00,0.000000e+00,0.0
Fwd Avg Bytes/Bulk,int64,1,0.000000,0.000000e+00,0.000000e+00,0.0
Bwd Avg Bulk Rate,int64,1,0.000000,0.000000e+00,0.000000e+00,0.0
Bwd URG Flags,int64,1,0.000000,0.000000e+00,0.000000e+00,0.0
Fwd URG Flags,int64,2,0.000112,1.059513e-02,0.000000e+00,1.0
Fwd PSH Flags,int64,2,0.045780,2.090075e-01,0.000000e+00,1.0


In [5]:
# ============================================================
# EXACT DUPLICATE FEATURE COLUMNS
# ============================================================

print("=" * 70)
print("EXACT DUPLICATE FEATURE ANALYSIS")
print("=" * 70)

duplicate_feature_pairs = []

for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):

        col1 = feature_cols[i]
        col2 = feature_cols[j]

        if X[col1].equals(X[col2]):
            duplicate_feature_pairs.append(
                (col1, col2)
            )

print(
    "\nNumber of exact duplicate feature pairs:",
    len(duplicate_feature_pairs)
)

print("\nDuplicate feature pairs:")

for pair in duplicate_feature_pairs:
    print(
        f"{pair[0]}  ==  {pair[1]}"
    )

EXACT DUPLICATE FEATURE ANALYSIS

Number of exact duplicate feature pairs: 33

Duplicate feature pairs:
Total Fwd Packets  ==  Subflow Fwd Packets
Total Backward Packets  ==  Subflow Bwd Packets
Fwd PSH Flags  ==  SYN Flag Count
Bwd PSH Flags  ==  Bwd URG Flags
Bwd PSH Flags  ==  Fwd Avg Bytes/Bulk
Bwd PSH Flags  ==  Fwd Avg Packets/Bulk
Bwd PSH Flags  ==  Fwd Avg Bulk Rate
Bwd PSH Flags  ==  Bwd Avg Bytes/Bulk
Bwd PSH Flags  ==  Bwd Avg Packets/Bulk
Bwd PSH Flags  ==  Bwd Avg Bulk Rate
Fwd URG Flags  ==  CWE Flag Count
Bwd URG Flags  ==  Fwd Avg Bytes/Bulk
Bwd URG Flags  ==  Fwd Avg Packets/Bulk
Bwd URG Flags  ==  Fwd Avg Bulk Rate
Bwd URG Flags  ==  Bwd Avg Bytes/Bulk
Bwd URG Flags  ==  Bwd Avg Packets/Bulk
Bwd URG Flags  ==  Bwd Avg Bulk Rate
Fwd Header Length  ==  Fwd Header Length.1
Fwd Avg Bytes/Bulk  ==  Fwd Avg Packets/Bulk
Fwd Avg Bytes/Bulk  ==  Fwd Avg Bulk Rate
Fwd Avg Bytes/Bulk  ==  Bwd Avg Bytes/Bulk
Fwd Avg Bytes/Bulk  ==  Bwd Avg Packets/Bulk
Fwd Avg Bytes/Bulk  ==  Bw

In [6]:
# ============================================================
# CORRELATION ANALYSIS
# ============================================================

corr_matrix = X.corr().abs()

upper = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = (
    upper.stack()
    .sort_values(ascending=False)
)

high_corr_pairs = high_corr_pairs[
    high_corr_pairs >= 0.95
]

print(
    "Number of feature pairs with |correlation| >= 0.95:",
    len(high_corr_pairs)
)

print("\nTop highly correlated feature pairs:")

display(
    high_corr_pairs.head(50)
)

Number of feature pairs with |correlation| >= 0.95: 40

Top highly correlated feature pairs:


Fwd URG Flags                CWE Flag Count                 1.000000
Total Backward Packets       Subflow Bwd Packets            1.000000
Total Fwd Packets            Subflow Fwd Packets            1.000000
Fwd Packet Length Mean       Avg Fwd Segment Size           1.000000
Fwd Header Length            Fwd Header Length.1            1.000000
Fwd PSH Flags                SYN Flag Count                 1.000000
Bwd Packet Length Mean       Avg Bwd Segment Size           1.000000
Total Length of Bwd Packets  Subflow Bwd Bytes              1.000000
Total Length of Fwd Packets  Subflow Fwd Bytes              0.999999
Total Fwd Packets            Subflow Bwd Packets            0.999077
Subflow Fwd Packets          Subflow Bwd Packets            0.999077
Total Backward Packets       Subflow Fwd Packets            0.999077
Total Fwd Packets            Total Backward Packets         0.999077
Flow Duration                Fwd IAT Total                  0.998542
Flow IAT Max                 Fwd I

In [7]:
# ============================================================
# SUSPICIOUS FEATURE VALUES
# ============================================================

suspicious_features = [
    "min_seg_size_forward",
    "Fwd Header Length",
    "Fwd Header Length.1"
]

for col in suspicious_features:

    print("\n" + "=" * 70)
    print(col)

    print("Minimum:", X[col].min())
    print("Maximum:", X[col].max())

    print(
        "Negative values:",
        (X[col] < 0).sum()
    )

    print("\nMost negative values:")
    print(
        X[col]
        .nsmallest(10)
        .to_list()
    )


min_seg_size_forward
Minimum: -536870661
Maximum: 138
Negative values: 35

Most negative values:
[-536870661, -536870661, -536870660, -536870660, -536870660, -536870660, -536870660, -536870660, -536870660, -536870660]

Fwd Header Length
Minimum: -32212234632
Maximum: 4644908
Negative values: 35

Most negative values:
[-32212234632, -9663668122, -6442447920, -6442447920, -4831831506, -2147482640, -1929349973, -1073741320, -1073741320, -1073741320]

Fwd Header Length.1
Minimum: -32212234632
Maximum: 4644908
Negative values: 35

Most negative values:
[-32212234632, -9663668122, -6442447920, -6442447920, -4831831506, -2147482640, -1929349973, -1073741320, -1073741320, -1073741320]


In [8]:
# ============================================================
# INVESTIGATE NEGATIVE FEATURE ANOMALIES
# ============================================================

anomaly_mask = (
    (X["min_seg_size_forward"] < 0) |
    (X["Fwd Header Length"] < 0)
)

anomalies = df.loc[anomaly_mask].copy()

print("=" * 70)
print("NEGATIVE FEATURE ANOMALY ANALYSIS")
print("=" * 70)

print("\nNumber of anomalous rows:")
print(len(anomalies))

print("\nLabels:")
print(
    anomalies["Label"].value_counts()
)

print("\nFeature values:")

display(
    anomalies[
        [
            "min_seg_size_forward",
            "Fwd Header Length",
            "Fwd Header Length.1",
            "Label"
        ]
    ]
)

NEGATIVE FEATURE ANOMALY ANALYSIS

Number of anomalous rows:
35

Labels:
Label
BENIGN    35
Name: count, dtype: int64

Feature values:


,min_seg_size_forward,Fwd Header Length,Fwd Header Length.1,Label
699821,-83885313,-1929349973,-1929349973,BENIGN
699828,-83885313,-83885093,-83885093,BENIGN
699830,-83885313,-1006620072,-1006620072,BENIGN
699835,-83885313,-167770470,-167770470,BENIGN
699836,-83885313,-83885125,-83885125,BENIGN
699837,-83885313,-83885113,-83885113,BENIGN
699838,-83885313,-83885081,-83885081,BENIGN
699840,-83885313,-83885125,-83885125,BENIGN
699841,-83885313,-167770458,-167770458,BENIGN
699842,-83885313,-83885137,-83885137,BENIGN
